In [ ]:
import sys
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from utils import kernel_quantile_band_rankx

sys.path.append("external/heart_failure_detection")
sys.path.append("external/heart_failure_detection/src/model")
from external.heart_failure_detection.src.model.inception_ensemble import \
    InceptionEnsemble
from external.heart_failure_detection.src.transform.transform import Normalize

weights_dir = "external/heart_failure_detection/weights"
model = InceptionEnsemble(weights_dir=weights_dir)
normalizer = Normalize()

In [ ]:
ECHONEXT_BASE_PATH = "/dataset/physionet.org/files/echonext/1.1.0/"

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 512

split_files = {
    "no_split": "EchoNext_no_split_waveforms.npy",
    "train": "EchoNext_train_waveforms.npy",
    "val": "EchoNext_val_waveforms.npy",
    "test": "EchoNext_test_waveforms.npy",
}

In [ ]:
metadata = pd.read_csv(ECHONEXT_BASE_PATH + "echonext_metadata_100k.csv")
model.eval().to(device)


def load_waveforms(path: str, mmap_mode: str = "r"):
    arr = np.load(path, mmap_mode=mmap_mode)
    return arr


def preprocess_waveforms(raw_waveforms: np.ndarray, target_len: int = 5000):
    waveforms = np.transpose(raw_waveforms, (0, 3, 2, 1))
    # limb leads are linearly dependent, keep only last 8 leads (II, III and V1-V6)
    waveforms = waveforms[:, -8:, :, 0]

    n_samples, n_leads, orig_len = waveforms.shape
    x_old = np.arange(orig_len)
    x_new = np.linspace(0, orig_len - 1, target_len)
    resampled = np.zeros((n_samples, n_leads, target_len), dtype=np.float32)

    for i in tqdm(range(n_samples), desc="Resampling ECGs"):
        for j in range(n_leads):
            resampled[i, j, :] = np.interp(x_new, x_old, waveforms[i, j, :])

    return resampled  # (N, 8, target_len)


def normalize_waveforms(waveforms_np: np.ndarray, normalizer):
    waveforms = torch.tensor(waveforms_np, dtype=torch.float32)
    waveforms = normalizer(waveforms.permute(0, 2, 1)).permute(0, 2, 1)
    return waveforms


def predict_waveforms(
    waveforms: torch.Tensor, model, batch_size: int = 1024, device: str = "cuda"
):
    predictions_list = []
    n_samples = waveforms.shape[0]

    with torch.no_grad():
        for start in tqdm(range(0, n_samples, batch_size), desc="Running inference"):
            end = start + batch_size
            batch = waveforms[start:end].to(device, non_blocking=True)
            batch_pred = model(batch)  # e.g. (B, 5, 1)
            predictions_list.append(batch_pred.cpu())

    predictions = torch.cat(predictions_list, dim=0)
    return predictions


def process_split(
    split_name: str,
    filename: str,
    base_path: str,
    model,
    normalizer,
    batch_size: int,
    device: str,
):
    print(f"===== Processing {split_name} =====")
    raw = load_waveforms(base_path + filename, mmap_mode="r")
    wf_np = preprocess_waveforms(raw, target_len=5000)
    wf_tensor = normalize_waveforms(wf_np, normalizer)  # still on CPU
    preds = predict_waveforms(wf_tensor, model, batch_size, device)
    return preds


all_predictions = {}
for split, fname in split_files.items():
    preds = process_split(
        split_name=split,
        filename=fname,
        base_path=ECHONEXT_BASE_PATH,
        model=model,
        normalizer=normalizer,
        batch_size=batch_size,
        device=device,
    )
    all_predictions[split] = preds  # torch tensor on CPU

In [ ]:
CONFIG_PATH = Path("echo_settings.yaml")

with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

PLOT_LIMS = cfg["PLOT_LIMS"]

In [ ]:
def attach_predictions_to_metadata(
    metadata: pd.DataFrame, preds, prob_col: str = "ensemble"
):
    if hasattr(preds, "detach"):
        preds = preds.detach().cpu()
    p = torch.sigmoid(preds).mean(axis=1).numpy()  # average over ensemble members
    meta = metadata.copy()
    meta[prob_col] = p
    return meta


def plot_rankx_for_split(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: dict = None,
    column_units_map: dict = None,
    lims=None,
    n_grid: int = 400,
    qs_inner: float = 0.45,  # for median band
    qs_outer: float = 0.25,  # for wider band
):
    """
    Create one figure for a given split, with one panel per echo parameter.
    """
    if lims is None:
        lims = {}

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(14, 3.111), squeeze=False)
    axs = axs[0]  # since 1 row

    legend_handles, legend_labels = None, None

    for i, (ax, column) in enumerate(zip(axs, plot_cols)):
        sub = metadata.copy()
        sub = sub[~sub[column].isna()].reset_index(drop=True)

        x = sub[column].values
        y = sub[prob_col].values  # HF probabilities

        if column == "lvef_value":
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 5
        elif column == "pasp_value":
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 1
        else:
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 0.1

        # sort by x
        ind = np.argsort(x)
        x = x[ind]
        y = y[ind]

        # ----- Median curve (inner band) -----
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qs_inner, 1 - qs_inner), n_grid=n_grid, const=1.0
        )
        qm = (qlo + qhi) / 2
        ax.plot(xg_q, qm, linewidth=2.0, color="C0", zorder=2, label="Median")

        # ----- Wider quantile band -----
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qs_outer, 1 - qs_outer), n_grid=n_grid, const=1.0
        )
        ax.fill_between(
            xg_q,
            qlo,
            qhi,
            alpha=0.4,
            color="C0",
            zorder=1,
            label=f"{int(qs_outer*100)}--{int((1-qs_outer)*100)}% range",
            edgecolor="none",
        )

        ax.set_ylim(0, 1)
        if column_name_map is not None and column in column_name_map:
            ax.set_title(column_name_map[column], fontsize=13)
        else:
            ax.set_title(column.replace("_", " "), fontsize=13)
        if column_units_map is not None and column in column_units_map:
            ax.set_xlabel(f"{column_units_map[column]}")
        else:
            ax.set_xlabel(column.replace("_", " "))

        if i == 0:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
        else:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

        # ----- KDE density on twin axis -----
        ax2 = ax.twinx()

        sns.kdeplot(
            x,
            ax=ax2,
            color="gray",
            fill=True,
            alpha=0.3,
            edgecolor="gray",
            linewidth=0.0,
            label="Parameter density",
            zorder=0,
            bw_adjust=0.7,
        )

        ax2.set_yticks([])
        ax2.patch.set_visible(False)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

        # remove spines of the histogram axis
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.set_ylabel("")

        # main axis spines
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()
            legend_handles += handles2
            legend_labels += labels2

        colname = column_name_map.get(column, column)
        if PLOT_LIMS.get(colname) is not None:
            ax.set_xlim(PLOT_LIMS[colname])
            ax2.set_xlim(PLOT_LIMS[colname])
        else:
            lo, hi = np.percentile(x, [1, 99])
            ax.set_xlim(lo, hi)
            ax2.set_xlim(lo, hi)

    # single legend outside the grid
    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=3,
            frameon=False,
            fontsize=12,
        )

    # global labels / title
    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=13,
    )
    # fig.suptitle(f"{split_name} split", y=1.02, fontsize=13)
    plt.tight_layout(rect=[0.00, 0.01, 1, 0.9])

    return fig


plot_cols = (
    "lvef_value",
    "pasp_value",
    "ivs_measurement",
    "lvpw_measurement",
    "tr_max_velocity_value",
    # "atrial_rate",
    # "ventricular_rate",
    # "pr_interval",
    # "qrs_duration",
)

train_metadata = metadata[metadata["split"] == "train"].reset_index(drop=True)
val_metadata = metadata[metadata["split"] == "val"].reset_index(drop=True)
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)
no_split_metadata = metadata[metadata["split"] == "no_split"].reset_index(drop=True)


column_name_map = {
    "lvef_value": "LVEF",
    "pasp_value": "PASP",
    "ivs_measurement": "IVS",
    "lvpw_measurement": "PWT",
    "tr_max_velocity_value": "TRv",
}
column_units_map = {
    "lvef_value": "%",
    "pasp_value": "mmHg",
    "ivs_measurement": "cm",
    "lvpw_measurement": "cm",
    "tr_max_velocity_value": "m/s",
}
all_metadata = attach_predictions_to_metadata(
    metadata,
    torch.cat(
        [
            all_predictions["train"],
            all_predictions["val"],
            all_predictions["test"],
            all_predictions["no_split"],
        ],
        dim=0,
    ),
    prob_col="ensemble",
)
fig = plot_rankx_for_split(
    all_metadata.query("most_recent_ecg == 1"),
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)
plt.savefig("figures/png/rankx_kernel_columbia.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_columbia.pgf", bbox_inches="tight")
plt.show()

In [ ]:
all_metadata.columns

In [ ]:
def plot_rankx_for_split_by_sex(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    male_value: str = "male",
    female_value: str = "female",
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: dict = None,
    column_units_map: dict = None,
    lims=None,
    n_grid: int = 400,
    qs_inner: float = 0.45,  # for median band
    qs_outer: float = 0.25,  # for wider band
):
    """
    Like plot_rankx_for_split, but draws two smooth curves and bands
    (male and female) in the same axes.
    """
    if lims is None:
        lims = {}

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5), squeeze=False)
    axs = axs[0]  # single row

    legend_handles, legend_labels = None, None

    # Colors per sex
    sex_specs = [
        (female_value, "C1", "Female"),
        (male_value, "C0", "Male"),
    ]

    for i, (ax, column) in enumerate(zip(axs, plot_cols)):
        sub = metadata.copy()
        sub = sub[~sub[column].isna()].reset_index(drop=True)

        if sub.empty:
            ax.set_visible(False)
            continue

        # Base x (all patients) for x-limits and density
        x_all = sub[column].values

        # jitter by column (same jitter used both sexes so they stay aligned)
        if column == "lvef_value":
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 5
        elif column == "pasp_value":
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 1
        else:
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 0.1

        x_all_j = x_all + jitter

        # x-limits
        if lims.get(column) is not None:
            ax.set_xlim(lims[column])
        else:
            lo, hi = np.percentile(x_all_j, [1, 99])
            ax.set_xlim(lo, hi)

        # ----- Sex-specific curves and bands -----
        for sex_val, color, sex_label in sex_specs:
            sub_sex = sub[sub[sex_col] == sex_val]
            if sub_sex.empty:
                continue

            # use same jitter indices for each sex
            idx_sex = sub.index.isin(sub_sex.index)
            x = x_all_j[idx_sex]
            y = sub.loc[idx_sex, prob_col].values

            # sort by x (not strictly necessary for the smoother, but keeps grids nice)
            ind = np.argsort(x)
            x = x[ind]
            y = y[ind]

            # Inner "median" band: we use mid of qs_inner band as a smooth median
            xg_q, qlo, qhi = kernel_quantile_band_rankx(
                x, y, qs=(qs_inner, 1 - qs_inner), n_grid=n_grid, const=1.0
            )
            qm = (qlo + qhi) / 2
            ax.plot(
                xg_q,
                qm,
                linewidth=2.0,
                color=color,
                zorder=3,
                label=f"{sex_label} median",
            )

            # Outer band
            xg_q, qlo, qhi = kernel_quantile_band_rankx(
                x, y, qs=(qs_outer, 1 - qs_outer), n_grid=n_grid, const=1.0
            )
            ax.fill_between(
                xg_q,
                qlo,
                qhi,
                alpha=0.25,
                color=color,
                zorder=2,
                label=f"{sex_label} {int(qs_outer*100)}–{int((1-qs_outer)*100)}% range",
                edgecolor="none",
            )

        ax.set_ylim(0, 1)

        # Titles / x-labels
        if column_name_map is not None and column in column_name_map:
            ax.set_title(column_name_map[column])
        else:
            ax.set_title(column.replace("_", " "))

        if column_units_map is not None and column in column_units_map:
            ax.set_xlabel(column_units_map[column])
        else:
            ax.set_xlabel(column.replace("_", " "))

        # y-axis (probability as percentage)
        if i == 0:
            ax.set_yticks(
                [0, 0.25, 0.5, 0.75, 1.0],
                ["0", "25", "50", "75", "100"],
            )
        else:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

        # ----- KDE density (all patients) on twin axis -----
        ax2 = ax.twinx()
        sns.kdeplot(
            x_all_j,
            ax=ax2,
            color="gray",
            fill=True,
            alpha=0.3,
            edgecolor="gray",
            linewidth=0.0,
            label="Parameter density",
            zorder=0,
            bw_adjust=0.5,
        )

        ax2.set_yticks([])
        ax2.patch.set_visible(False)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

        # remove spines of the histogram axis
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.set_ylabel("")

        # main axis spines
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # collect legend entries from the first panel
        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()
            legend_handles += handles2
            legend_labels += labels2

    # single legend outside the grid
    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=5,
            frameon=False,
            fontsize=11,
        )

    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=11,
    )
    plt.tight_layout(rect=[0.00, 0.01, 1, 0.9])

    return fig


fig = plot_rankx_for_split_by_sex(
    # all_metadata,
    all_metadata.query("most_recent_ecg == 1"),
    prob_col="ensemble",
    sex_col="sex",
    male_value="male",
    female_value="female",
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)
plt.savefig("figures/png/rankx_kernel_columbia_male_female.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_columbia_male_female.pgf", bbox_inches="tight")
plt.show()

In [ ]:
def _spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    return np.abs(spearmanr(x, y).correlation)


def plot_rankx_by_race_and_sex(
    metadata: pd.DataFrame,
    plot_cols: List[str],
    races: Optional[List[str]] = None,
    race_col: str = "race_ethnicity",
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    sexes: Sequence[str] = ("female", "male"),  # order of the two bars per race
    column_name_map: Optional[Dict[str, str]] = None,
    column_units_map: Optional[Dict[str, str]] = None,
    qs_inner: float = 0.95,  # upper quantile (e.g. 95th)
    qs_outer: float = 0.05,  # lower quantile (e.g. 5th)
    n_boot: int = 10,
    random_state: Optional[int] = None,
    min_n: int = 10,  # minimum samples per race/sex/column to attempt bootstrapping
):
    """
    For each column in plot_cols, bootstrap the Spearman correlation between that
    column and prob_col, separately within each race *and* sex, and plot the median
    and 5–95% percentile intervals as grouped error bars.

    One subplot per column; within each subplot, x-axis = races, and for each race
    there are two points (female, male).
    """
    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    # Determine race order
    if races is None:
        races = metadata[race_col].dropna().astype(str).value_counts().index.tolist()

    rng = np.random.default_rng(random_state)

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), squeeze=False)
    axs = axs[0]

    x_positions = np.arange(len(races))  # one center x per race
    n_sexes = len(sexes)
    width = 0.25  # half-width of the race "group"
    # symmetric offsets around the race center for each sex
    if n_sexes == 2:
        offsets = np.array([-width / 2, width / 2])
    else:
        offsets = np.linspace(-width, width, n_sexes)

    for ax, column in zip(axs, plot_cols):
        medians = np.full((n_sexes, len(races)), np.nan, dtype=float)
        los = np.full_like(medians, np.nan)
        his = np.full_like(medians, np.nan)
        ns = np.zeros_like(medians, dtype=int)

        for s_idx, sex in enumerate(sexes):
            for r_idx, race in enumerate(races):
                sub = metadata.loc[
                    (metadata[race_col] == race) & (metadata[sex_col] == sex),
                    [column, prob_col],
                ]
                sub = sub.dropna().reset_index(drop=True)

                x = sub[column].to_numpy()
                y = sub[prob_col].to_numpy()
                n = len(x)
                ns[s_idx, r_idx] = n

                if n < min_n:
                    continue  # remains NaN

                boot_corrs = np.empty(n_boot, dtype=float)
                for b in range(n_boot):
                    idx = rng.integers(0, n, size=n)
                    boot_corrs[b] = _spearman_corr(x[idx], y[idx])

                boot_corrs = boot_corrs[~np.isnan(boot_corrs)]
                if len(boot_corrs) == 0:
                    continue

                medians[s_idx, r_idx] = np.median(boot_corrs)
                los[s_idx, r_idx] = np.quantile(boot_corrs, qs_outer)
                his[s_idx, r_idx] = np.quantile(boot_corrs, qs_inner)

        # Plot sex-specific errorbars with small x offsets
        for s_idx, sex in enumerate(sexes):
            sex_medians = medians[s_idx]
            sex_los = los[s_idx]
            sex_his = his[s_idx]

            valid = ~np.isnan(sex_medians)
            if not np.any(valid):
                continue

            yerr = np.vstack(
                [
                    sex_medians[valid] - sex_los[valid],
                    sex_his[valid] - sex_medians[valid],
                ]
            )

            ax.errorbar(
                x_positions[valid] + offsets[s_idx],
                sex_medians[valid],
                yerr=yerr,
                fmt="o",
                capsize=4,
                label=str(sex),
                color=f"C{3-s_idx*3}",
            )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(races, rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("Spearman ρ")
        ax.set_xlim(x_positions[0] - 0.75, x_positions[-1] + 0.75)

        pretty = column_name_map.get(column, column)
        unit = column_units_map.get(column, "")
        if unit:
            title = f"{pretty} [{unit}]"
        else:
            title = pretty
        ax.set_title(title, fontsize=11)
        ax.legend(title=sex_col)

    fig.tight_layout()
    return fig, axs

In [ ]:
races = ["black", "white", "asian", "hispanic"]

fig, axs = plot_rankx_by_race_and_sex(
    # all_metadata,
    all_metadata.query(
        "most_recent_ecg == 1 and race_ethnicity != 'unknown' and race_ethnicity != 'other'"
    ),
    plot_cols=plot_cols,
    races=races,
    race_col="race_ethnicity",
    prob_col="ensemble",
    sex_col="sex",
    sexes=("female", "male"),
    column_name_map=column_name_map,
    column_units_map=column_units_map,
    qs_inner=0.975,
    qs_outer=0.025,
    n_boot=500,
    random_state=42,
    min_n=10,
)

plt.show()

In [ ]:
from typing import Dict, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def plot_rankx_by_sex_and_race(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    race_col: str = "race_ethnicity",
    male_value: str = "male",
    female_value: str = "female",
    races: Optional[List[str]] = None,  # order of races
    sexes: Sequence[str] = ("female", "male"),  # order of sexes → rows
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: Dict[str, str] = None,
    column_units_map: Dict[str, str] = None,
    lims=None,
    n_grid: int = 400,
    qs_median_band: float = 0.35,  # used to compute a robust median curve
):
    """
    Stratify by both sex and race_ethnicity.

    - Two rows: one per sex (first females, then males).
    - Columns are measurement variables (plot_cols).
    - Color encodes race.
    - Only a single smooth 'median' curve per race group (no IQR bands).
    - KDEs are separate per sex (row).
    """

    if lims is None:
        lims = {}
    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    # Determine race order if not provided
    if races is None:
        races = metadata[race_col].dropna().astype(str).value_counts().index.tolist()

    # Map race → color
    race_colors = {race: f"C{i % 10}" for i, race in enumerate(races)}

    n_cols = len(plot_cols)
    n_rows = len(sexes)

    # Two rows (sexes) × columns (measurements)
    fig, axs = plt.subplots(
        n_rows,
        n_cols,
        figsize=(14, 5.5),
        squeeze=False,
    )

    legend_entries = {}

    for j, column in enumerate(plot_cols):
        # All non-missing values for this column to set consistent x-limits
        sub_all = metadata[~metadata[column].isna()].reset_index(drop=True)
        if sub_all.empty:
            # hide entire column if no data at all
            for i in range(n_rows):
                axs[i, j].set_visible(False)
            continue

        x_all = sub_all[column].values

        # jitter for x-limits (based on all patients)
        if column == "lvef_value":
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 5
        elif column == "pasp_value":
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 1
        else:
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 0.1

        x_all_j_all = x_all + jitter_all

        # x-limits shared across sexes for this column
        if lims.get(column) is not None:
            x_lo, x_hi = lims[column]
        else:
            x_lo, x_hi = np.percentile(x_all_j_all, [1, 99])

        for i, sex in enumerate(sexes):
            ax = axs[i, j]

            # restrict to this sex
            sub = sub_all[sub_all[sex_col] == sex].reset_index(drop=True)
            if sub.empty:
                ax.set_visible(False)
                continue

            # Base x for this sex
            x_sex = sub[column].values
            if column == "lvef_value":
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 5
            elif column == "pasp_value":
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 1
            else:
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 0.1

            x_sex_j = x_sex + jitter

            ax.set_xlim(x_lo, x_hi)

            # ----- Race-specific curves (no bands) for this sex -----
            for race in races:
                sub_race = sub[sub[race_col] == race]
                if sub_race.empty:
                    continue

                color = race_colors[race]

                idx_race = (sub[race_col] == race).values
                x = x_sex_j[idx_race]
                y = sub.loc[idx_race, prob_col].values

                # sort by x for smoother grid
                ind = np.argsort(x)
                x = x[ind]
                y = y[ind]

                # Use inner quantile band to approximate a robust median curve
                xg_q, qlo, qhi = kernel_quantile_band_rankx(
                    x,
                    y,
                    qs=(qs_median_band, 1 - qs_median_band),
                    n_grid=n_grid,
                    const=2.0,
                )
                qm = (qlo + qhi) / 2.0  # robust "median" curve

                label = race  # no sex in label; rows are sexes

                line = ax.plot(
                    xg_q,
                    qm,
                    linewidth=2.0,
                    color=color,
                    linestyle="-",  # all solid; sex is encoded by rows
                    zorder=3,
                    label=label,
                )[0]

                # store a single handle per race for legend
                if label not in legend_entries:
                    legend_entries[label] = line

            ax.set_ylim(0, 1)

            # Titles / x-labels
            if column in column_name_map:
                ax.set_title(column_name_map[column], fontsize=13)
            else:
                ax.set_title(column.replace("_", " "), fontsize=13)

            if column in column_units_map:
                ax.set_xlabel(column_units_map[column], fontsize=11)
            else:
                ax.set_xlabel(column.replace("_", " "), fontsize=11)

            # y-axis (probability as percentage)
            if j == 0:
                # leftmost column shows y-ticks for both rows
                ax.set_yticks(
                    [0, 0.25, 0.5, 0.75, 1.0],
                    ["0", "25", "50", "75", "100"],
                )
            else:
                ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

            # annotate row with sex on the left of the first column
            if j == 0:
                ax.text(
                    -0.25,
                    0.5,
                    sex.capitalize(),
                    transform=ax.transAxes,
                    rotation=90,
                    va="center",
                    ha="center",
                    fontsize=11,
                )

            # ----- KDE density (this sex only) on twin axis -----
            ax2 = ax.twinx()
            sns.kdeplot(
                x_sex_j,
                ax=ax2,
                color="gray",
                fill=True,
                alpha=0.3,
                edgecolor="gray",
                linewidth=0.0,
                zorder=0,
                bw_adjust=0.6,
            )

            ax2.set_yticks([])
            ax2.patch.set_visible(False)
            ax.set_zorder(ax2.get_zorder() + 1)
            ax.patch.set_visible(False)

            # remove spines of the histogram axis
            ax2.spines["top"].set_visible(False)
            ax2.spines["right"].set_visible(False)
            ax2.spines["bottom"].set_visible(False)
            ax2.spines["left"].set_visible(False)
            ax2.set_ylabel("")

    # single legend outside the grid: races only
    if legend_entries:
        handles = list(legend_entries.values())
        labels = list(legend_entries.keys())
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=min(len(labels), 4),
            frameon=False,
            fontsize=12,
        )

    # global y-label
    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=12,
    )

    # clean up spines on main axes
    for row_axes in axs:
        for ax in row_axes:
            if not ax.get_visible():
                continue
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    # remove x label, x ticks frop top row, and title from bottom row
    for j in range(n_cols):
        ax_top = axs[0, j]
        ax_bottom = axs[1, j]
        if ax_top.get_visible():
            ax_top.set_xlabel("")
            ax_top.set_xticklabels([])
        if ax_bottom.get_visible():
            ax_bottom.set_title("")
    plt.tight_layout(rect=[0.015, 0.00, 1, 0.94])

    return fig


fig = plot_rankx_by_sex_and_race(
    all_metadata.query(
        "most_recent_ecg == 1 and race_ethnicity != 'unknown' and race_ethnicity != 'other'"
    ),
    prob_col="ensemble",
    sex_col="sex",
    race_col="race_ethnicity",
    male_value="male",
    female_value="female",
    races=None,
    sexes=("male", "female"),
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)

plt.savefig("figures/png/rankx_kernel_by_race_sex_columbia.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_by_race_sex_columbia.pgf", bbox_inches="tight")
plt.show()

In [ ]:
# Build the counts table
counts = (
    all_metadata.query("most_recent_ecg == 1")
    .dropna(subset=["race_ethnicity", "sex"])
    .groupby(["race_ethnicity", "sex"])
    .size()
    .reset_index(name="n_patients")
)

table = (
    counts.pivot(index="race_ethnicity", columns="sex", values="n_patients")
    .fillna(0)
    .astype(int)
)

# Capitalize sex column names
table.columns = [col.capitalize() for col in table.columns]

# Add Total column BEFORE Male and Female
table["Total"] = table.sum(axis=1)

# Reorder columns: Total, Male, Female (only those that exist)
ordered_cols = ["Total"] + [c for c in ["Male", "Female"] if c in table.columns]
table = table[ordered_cols]

# Sort rows by Total (descending)
table = table.sort_values(by="Total", ascending=False)

# -----------------------------------------
# Build LaTeX table manually (with \num{})
# -----------------------------------------

lines = []

lines.append("\\begin{table}")
lines.append("    \\centering")
lines.append("    \\begin{threeparttable}")
lines.append("    \\caption{Counts of patients by race and sex.}")
lines.append("    \\label{tab:race_sex_counts}")

# tabular environment
col_spec = "l" + "r" * len(table.columns)
lines.append("    \\begin{tabular}{" + col_spec + "}")
lines.append("        \\toprule")

# Header row
header = "Race"
for col in table.columns:
    header += " & " + str(col)
header += " \\\\"
lines.append("        " + header)
lines.append("        \\midrule")

# Data rows with \num{X}
for race in table.index:
    row = race.capitalize()
    for col in table.columns:
        val = table.loc[race, col]
        row += " & \\num{" + str(val) + "}"
    row += " \\\\"
    lines.append("        " + row)

lines.append("        \\bottomrule")
lines.append("    \\end{tabular}")

lines.append("    \\end{threeparttable}")
lines.append("\\end{table}")

latex_output = "\n".join(lines)
print(latex_output)

In [ ]:
metadata["lvef_lte_45_flag"].value_counts()
all_preds_concated = torch.cat(
    [
        all_predictions["train"],
        all_predictions["val"],
        all_predictions["test"],
        all_predictions["no_split"],
    ],
    dim=0,
)
auc = roc_auc_score(
    metadata["lvef_lte_45_flag"], torch.sigmoid(all_preds_concated).mean(axis=1).numpy()
)
print("AUC for shd_moderate_or_greater_flag:", auc)

In [ ]:
for column in plot_cols:
    print(column)
    sub = all_metadata.query("most_recent_ecg == 1").copy()
    sub = sub[~sub[column].isna()].reset_index(drop=True)

    x = sub[column].values
    y = sub["ensemble"].values  # HF probabilities

    corr, pval = spearmanr(x, y)
    print(
        f"    Spearman correlation between {column} and ensemble: {corr:.4f} (p={pval:.2e})"
    )